In [1]:
import numpy as np
import pandas as pd

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)


In [2]:
monitor = pd.read_csv("/workspaces/CUBES/Eplus-env-09-23_thermostat_rbc_eco_year_2023_case_0_rep_0_zone_9_onoffseed_5_tempseed_30_manualsetbacktemp_17_comforttemp_20_setbacktemp_15_paper2_final_runs_v3-res1/Eplus-env-sub_run1/monitor.csv")

/tmp/ipykernel_2601/3453730822.py:1: DtypeWarning: Columns (133,140) have mixed types. Specify dtype option on import or set low_memory=False.
  monitor = pd.read_csv("/workspaces/CUBES/Eplus-env-09-23_thermostat_rbc_eco_year_2023_case_0_rep_0_zone_9_onoffseed_5_tempseed_30_manualsetbacktemp_17_comforttemp_20_setbacktemp_15_paper2_final_runs_v3-res1/Eplus-env-sub_run1/monitor.csv")


In [5]:
h46 = pd.read_csv("/workspaces/CUBES/exp/jack/misc/increasing_timestep/H46_cleaned_2013.csv")


# Assuming your dataframe is named df and the 'UTC_Time' column is already in datetime format
h46['UTC_Time'] = pd.to_datetime(h46['UTC_Time'])

mapping = {'H46_Backroom(Down stairs)': 'backroom',
           'H46_Bathroom 1(Upstairs)': 'bathroom',
           'H46_Hall(Down stairs)':'hall_downstairs',
           'H46_Front Room(Down stairs)':'front_room',
           'H46_Kitchen(Down stairs)':'kitchen',
           'H46_Bedroom 3(Upstairs)':'bedroom_3',
            'H46_Bedroom 1(Upstairs)': 'bedroom_1',
            'H46_Bedroom 2(Upstairs)': 'bedroom_2',
           }

h46 = h46.rename(columns=mapping)

h46 = h46.drop(columns=['H46_Bedroom 5(Upstairs)', 'H46_Utility(Down stairs)'])

h46["hall_upstairs"] = h46["hall_downstairs"]



In [8]:

# Assuming df is your original DataFrame
df_copy = h46.copy()

# Convert 'UTC_Time' to datetime to work with times
df_copy['UTC_Time'] = pd.to_datetime(df_copy['UTC_Time'])

# Create a boolean mask for the time between 23:00 and 07:00
time_mask = (df_copy['UTC_Time'].dt.hour >= 23) | (df_copy['UTC_Time'].dt.hour < 7)

# Define a function to apply the heating rule
def apply_heating_rule(series, time_mask):
    heating = np.zeros_like(series)  # Initial heating schedule, all off (0)
    for i in range(5, len(series)):  # Start from 5th element due to 5-minute window
        if time_mask[i]:  # If time is between 23:00 and 07:00, heating is off
            heating[i] = 0
        elif series[i] > 0:  # If current value is greater than 0, heating is on for 5 mins
            heating[i-4:i+1] = 1
        elif np.all(series[i-4:i] == 0):  # If no activity in the last 5 mins, heating off
            heating[i] = 0
    return heating

# Apply the heating rule to all rooms
rooms = ['backroom', 'bathroom', 'front_room', 'hall_downstairs', 'bedroom_2', 'kitchen', 'bedroom_1', 'bedroom_3', 'hall_upstairs']

for room in rooms:
    df_copy[room] = apply_heating_rule(h46[room].values, time_mask)

# Now df_copy contains your desired heating schedule


In [9]:
df_copy


,UTC_Time,backroom,bathroom,front_room,hall_downstairs,bedroom_2,kitchen,bedroom_1,bedroom_3,hall_upstairs
0,2013-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2013-01-01 00:01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2013-01-01 00:02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2013-01-01 00:03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2013-01-01 00:04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525596,2013-12-31 23:56:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525597,2013-12-31 23:57:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525598,2013-12-31 23:58:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# Resample the dataframe in 10-minute intervals, applying the max function to each group
df_resampled = df_copy.resample('10T', on='UTC_Time').max()

# If you need to reset the index to make 'UTC_Time' a normal column again
df_resampled = df_resampled.reset_index()

df_resampled = df_resampled.set_index("UTC_Time")
df_resampled = df_resampled.clip(upper=1)
df_resampled.to_csv("heating_pattern_5min_h46.sch")

/tmp/ipykernel_2601/2944430544.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df_copy.resample('10T', on='UTC_Time').max()


In [4]:
df_resampled

,backroom,bathroom,front_room,hall_downstairs,bedroom_2,kitchen,bedroom_1,bedroom_3,hall_upstairs
UTC_Time,,,,,,,,,
2013-01-01 00:00:00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2013-01-01 00:10:00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2013-01-01 00:20:00,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2013-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01 00:40:00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
2013-12-31 23:10:00,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2013-12-31 23:20:00,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2013-12-31 23:30:00,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
# Resample the dataframe in 10-minute intervals, applying the max function to each group
df_resampled = h46.resample('10T', on='UTC_Time').max()

# If you need to reset the index to make 'UTC_Time' a normal column again
df_resampled = df_resampled.reset_index()



df_resampled = df_resampled.set_index("UTC_Time")
df_resampled = df_resampled.clip(upper=1)
df_resampled.to_csv("schedule_h46.sch")

In [ ]:
df_resampled

In [69]:
monitor = pd.read_csv("/workspaces/CUBES/Eplus-env-09-23_thermostat_rbc_eco_year_2023_case_0_rep_0_zone_9_onoffseed_5_tempseed_30_manualsetbacktemp_17_comforttemp_20_setbacktemp_15_paper2_final_runs_v3-res2/Eplus-env-sub_run1/monitor.csv")

In [70]:
monitor = monitor[(monitor["hour"]> 6) & (monitor["hour"] < 23)]

In [72]:
monitor.filter(like="kitchen")

,Schedule Value(Heating-Pattern-Schedule-kitchen),Zone Operative Temperature(kitchen),Zone Air Temperature(kitchen),Zone Air Temperature(kitchen).1,Zone Air Relative Humidity(kitchen),Zone Air CO2 Concentration(kitchen),Zone People Occupant Count(kitchen),Schedule Value(kitchen-Thermostat Dual SP Control-HEATING-EXT),Zone Ventilation Air Change Rate(kitchen),Schedule Value(kitchen-Ventilation-EXT),kitchen-Ventilation-EXT,kitchen-Thermostat Dual SP Control-HEATING-EXT
42,0.0,16.790680,16.594620,16.594620,49.953820,420.16406,0.0,15.0,0.000000,0.0,0.0,15.0
43,1.0,16.759537,16.563475,16.563475,50.087450,420.13287,0.0,15.0,0.000000,0.0,0.0,15.0
44,1.0,18.740700,19.999012,19.999012,40.384330,420.11734,0.0,20.0,0.000000,0.0,0.0,20.0
45,1.0,18.861160,19.999853,19.999853,44.592660,675.39026,4.0,20.0,0.000000,0.0,0.0,20.0
46,1.0,19.066593,20.263018,20.263018,46.158566,814.42460,0.0,20.0,0.000000,0.0,0.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2149,0.0,15.667466,15.026023,15.026023,44.244682,727.16750,0.0,15.0,2.079135,1.0,1.0,15.0
2150,1.0,16.014078,15.826355,15.826355,43.036144,833.25220,4.0,15.0,2.084223,1.0,1.0,15.0
2151,1.0,18.407333,19.920683,19.920683,32.973730,805.95526,0.0,20.0,2.114292,1.0,1.0,20.0
2152,1.0,18.551691,19.999783,19.999783,31.334902,633.01953,0.0,20.0,2.114867,1.0,1.0,20.0


In [47]:
schedule = pd.read_csv("/workspaces/CUBES/cubes/data/schedules/thermostat_exp/rep0/schedule_rep_0.sch")

In [52]:
schedule.columns

Index(['Unnamed: 0', 'hall_downstairs', 'front_room', 'kitchen', 'backroom',
       'bedroom_3', 'bedroom_1', 'hall_upstairs', 'bathroom', 'bedroom_2'],
      dtype='object')

In [57]:
df = pd.read_csv("/workspaces/CUBES/exp/jack/misc/increasing_timestep/heating_commands.csv")

In [58]:
df = df.drop(columns=['H46_Bedroom 5(Upstairs)', 'H46_Utility(Down stairs)'])

In [59]:
df

,UTC_Time,H46_Backroom(Down stairs),H46_Bathroom 1(Upstairs),H46_Front Room(Down stairs),H46_Hall(Down stairs),H46_Bedroom 2(Upstairs),H46_Kitchen(Down stairs),H46_Bedroom 1(Upstairs),H46_Bedroom 3(Upstairs)
0,2013-01-01 00:00:00,0,0,0,0,0,0,0,0
1,2013-01-01 00:01:00,0,0,0,0,0,0,0,0
2,2013-01-01 00:02:00,0,0,0,0,0,0,0,0
3,2013-01-01 00:03:00,0,0,0,0,0,0,0,0
4,2013-01-01 00:04:00,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0,0,0,0,0,0,0,0
525596,2013-12-31 23:56:00,0,0,0,0,0,0,0,0
525597,2013-12-31 23:57:00,0,0,0,0,0,0,0,0
525598,2013-12-31 23:58:00,0,0,0,0,0,0,0,0


In [62]:
mapping = {'H46_Backroom(Down stairs)': 'backroom',
           'H46_Bathroom 1(Upstairs)': 'bathroom',
           'H46_Hall(Down stairs)':'hall_downstairs',
           'H46_Front Room(Down stairs)':'front_room',
           'H46_Kitchen(Down stairs)':'kitchen',
           'H46_Bedroom 3(Upstairs)':'bedroom_3',
            'H46_Bedroom 1(Upstairs)': 'bedroom_1',
            'H46_Bedroom 2(Upstairs)': 'bedroom_2',
           }

df = df.rename(columns=mapping)

In [64]:
df["hall_upstairs"] = df["hall_downstairs"]

In [65]:
df

,UTC_Time,backroom,bathroom,front_room,hall_downstairs,bedroom_2,kitchen,bedroom_1,bedroom_3,hall_upstairs
0,2013-01-01 00:00:00,0,0,0,0,0,0,0,0,0
1,2013-01-01 00:01:00,0,0,0,0,0,0,0,0,0
2,2013-01-01 00:02:00,0,0,0,0,0,0,0,0,0
3,2013-01-01 00:03:00,0,0,0,0,0,0,0,0,0
4,2013-01-01 00:04:00,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0,0,0,0,0,0,0,0,0
525596,2013-12-31 23:56:00,0,0,0,0,0,0,0,0,0
525597,2013-12-31 23:57:00,0,0,0,0,0,0,0,0,0
525598,2013-12-31 23:58:00,0,0,0,0,0,0,0,0,0


In [66]:
# Assuming your dataframe is named df and the 'UTC_Time' column is already in datetime format
df['UTC_Time'] = pd.to_datetime(df['UTC_Time'])

# Resample the dataframe in 10-minute intervals, applying the max function to each group
df_resampled = df.resample('10T', on='UTC_Time').max()

# If you need to reset the index to make 'UTC_Time' a normal column again
df_resampled = df_resampled.reset_index()

/tmp/ipykernel_10264/2487096361.py:5: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('10T', on='UTC_Time').max()


In [68]:
df_resampled.to_csv("heating_pattern_h46.sch")

### Extending occupancy with random numbers in between

In [19]:

# Load the schedule file
df = pd.read_csv("/workspaces/CUBES/cubes/data/schedules/thermostat_exp/rep0/schedule_rep_0.sch", index_col=0)

# Convert the index to datetime format, specifying dayfirst=True for European format dates
df.index = pd.to_datetime(df.index, dayfirst=True)

# Resample the data to 1-minute frequency and fill with NaNs
df_resampled = df.resample('1T').asfreq()

# Replace NaNs with random 0 or 1
df_resampled = df_resampled.apply(lambda x: np.random.choice([0, 1], size=len(x)) if pd.api.types.is_numeric_dtype(x) else x)

# Format the index to match "dd/mm/YYYY HH:MM"
df_resampled.index = df_resampled.index.strftime("%d/%m/%Y %H:%M")

# Save the new dataset to CSV
df_resampled.to_csv('resampled_minute_data.sch')


/tmp/ipykernel_65717/1975152266.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('1T').asfreq()


### Extending electricity import and export price, grid emissions intensity, gas price

In [16]:

# Define the function to load, resample, and save the data
def resample_and_forward_fill(file_path, output_path):
    # Load the CSV file with the timestamp column as index
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)

    # Resample the data to 1-minute frequency and forward-fill missing values
    df_resampled = df.resample('1T').ffill()

    # Save the resampled data to a new CSV
    df_resampled.to_csv(output_path)

# File paths
elect_import = "/workspaces/CUBES/cubes/data/electricity/csv_agile_A_Eastern_England_2023.csv"
elect_export = "/workspaces/CUBES/cubes/data/electricity_export/csv_agileoutgoing_A_Eastern_England_2023.csv"
gas_import = "/workspaces/CUBES/cubes/data/gas/csv_gastracker_A_Eastern_England_2023.csv"
grid = "/workspaces/CUBES/cubes/data/grid/grid_carbon_GB_10min_2023.csv"

# Output paths (can be the same as the input or different if you prefer)
output_elect_import = "/workspaces/CUBES/cubes/data/electricity/resampled_csv_agile_A_Eastern_England_2023.csv"
output_elect_export = "/workspaces/CUBES/cubes/data/electricity_export/resampled_csv_agileoutgoing_A_Eastern_England_2023.csv"
output_gas_import = "/workspaces/CUBES/cubes/data/gas/resampled_csv_gastracker_A_Eastern_England_2023.csv"
output_grid = "/workspaces/CUBES/cubes/data/grid/resampled_grid_carbon_GB_10min_2023.csv"

# Resample and forward fill each file
resample_and_forward_fill(elect_import, output_elect_import)
resample_and_forward_fill(elect_export, output_elect_export)
resample_and_forward_fill(gas_import, output_gas_import)
resample_and_forward_fill(grid, output_grid)


/tmp/ipykernel_65717/197032183.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('1T').ffill()
/tmp/ipykernel_65717/197032183.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('1T').ffill()
/tmp/ipykernel_65717/197032183.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('1T').ffill()
/tmp/ipykernel_65717/197032183.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('1T').ffill()


### Extending weather

In [15]:
import pandas as pd

# Define a function to process .epw files
def process_epw(file_path, output_path):
    # Read the file and extract the data starting from the actual weather data (usually after 8 lines)
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Metadata part remains unchanged (first 8 lines)
    metadata = lines[:8]

    # The data part starts from line 9 onwards
    data_lines = lines[8:]

    # Parse the weather data into a DataFrame, splitting the data into columns
    data = [line.strip().split(',') for line in data_lines]
    df = pd.DataFrame(data)

    # Create a datetime index for resampling, combining year, month, day, hour
    df[0] = df[0].astype(str)  # Year
    df[1] = df[1].astype(str)  # Month
    df[2] = df[2].astype(str)  # Day
    df[3] = df[3].astype(int)  # Hour

    # Adjust for hours that are 24 by setting them to 0 and incrementing the date
    df.loc[df[3] == 24, 3] = 0
    df['datetime'] = pd.to_datetime(df[0] + '-' + df[1] + '-' + df[2], format='%Y-%m-%d') + pd.to_timedelta(df[3], unit='h')

    df.set_index('datetime', inplace=True)

    # Resample to a higher frequency (e.g., 10-minute intervals)
    df_resampled = df.resample('10T').ffill()  # Forward-fill to fill missing values

    # Convert resampled DataFrame back into the format required for EPW
    df_resampled = df_resampled.reset_index()

    # Update year, month, day, and hour fields according to the new datetime index
    df_resampled[0] = df_resampled['datetime'].dt.year
    df_resampled[1] = df_resampled['datetime'].dt.month
    df_resampled[2] = df_resampled['datetime'].dt.day
    df_resampled[3] = df_resampled['datetime'].dt.hour

    # Prepare the final resampled data lines
    resampled_lines = df_resampled.drop(columns=['datetime']).astype(str).apply(lambda row: ','.join(row), axis=1).tolist()

    # Write back to a new EPW file
    with open(output_path, 'w') as output_file:
        output_file.writelines(metadata)  # Write the unchanged metadata
        output_file.write('\n'.join(resampled_lines))  # Write the new resampled data lines


# Example usage
epw_file = "/workspaces/CUBES/cubes/data/weather/Cambridgeshire_CC_2023.epw"
output_epw_file = "/workspaces/CUBES/cubes/data/weather/resampled_weather_file.epw"

process_epw(epw_file, output_epw_file)


/tmp/ipykernel_65717/3254003127.py:32: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df.resample('10T').ffill()  # Forward-fill to fill missing values
